# STEP 3. 데이터 불러오기와 품질 점검

- 데이터: Google Trends "해외여행" 검색 관심도 (대한민국, 주간, 2021-11-07 ~ 2026-11-08)
- 값: 0~100 상대 지수 (실제 검색 횟수가 아님)
- 출처 표기: Data source: Google Trends (https://www.google.com/trends)

## 처리 기준 (STEP 3에서 확정)

| 구분 | 기준 | 이유 |
|---|---|---|
| 결측치 | 있으면 채우지 않고 개수와 위치를 기록한다 | 상대 지수라 보간값은 근거가 약하다 |
| 마지막 주 | 통계에서 제외하고 원본 파일에는 유지한다 | 수집일(2026-11-09)에 끝나지 않은 주일 수 있다 |
| 이상치 | 13주 이동 중앙값에서 크게 벗어난 주를 표시만 하고 삭제하지 않는다 | 급증이 실제 관심 변화일 수 있고, 추세가 있어 전체 분포 기준은 부적합하다 |

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

print("Python :", sys.version.split()[0])
print("numpy  :", np.__version__)
print("pandas :", pd.__version__)

Python : 3.13.17
numpy  : 2.4.6
pandas : 2.2.3


## 1. 불러오기

원본 CSV는 1행이 카테고리, 2행이 빈 줄, 3행이 열 이름이다. 앞 2줄을 건너뛰고 읽는다.

In [2]:
# 로컬에서는 data 폴더의 파일을, 파일이 없으면(Colab 등) GitHub의 원본 주소를 사용한다
CSV_PATH = Path("data/google_trends_overseas_travel_raw.csv")
RAW_URL = (
    "https://raw.githubusercontent.com/geunwa/"
    "overseas-travel-trend-analysis/main/data/google_trends_overseas_travel_raw.csv"
)
source = CSV_PATH if CSV_PATH.exists() else RAW_URL

raw = pd.read_csv(source, skiprows=2, encoding="utf-8-sig")
raw.columns = ["date", "value"]
raw["date"] = pd.to_datetime(raw["date"])
raw = raw.sort_values("date").reset_index(drop=True)

print("읽은 곳:", source)
print("행, 열 :", raw.shape)
print(raw.head(3))
print(raw.tail(3))

읽은 곳: data\google_trends_overseas_travel_raw.csv
행, 열 : (262, 2)
        date  value
0 2021-11-07     25
1 2021-11-14     22
2 2021-11-21     20
          date  value
259 2026-10-25     82
260 2026-11-01     77
261 2026-11-08     74


## 2. 품질 점검

결측치, 중복 날짜, 주 간격, 요일, 0값, 값의 범위를 확인한다.

In [3]:
gaps = raw["date"].diff().dt.days.dropna()

checks = pd.DataFrame(
    {
        "점검 항목": [
            "행 수",
            "결측치 수",
            "중복 날짜 수",
            "주 간격이 7일이 아닌 곳",
            "일요일이 아닌 날짜",
            "값이 0인 주",
            "값 최솟값",
            "값 최댓값",
        ],
        "결과": [
            len(raw),
            int(raw["value"].isna().sum()),
            int(raw["date"].duplicated().sum()),
            int((gaps != 7).sum()),
            int((raw["date"].dt.dayofweek != 6).sum()),
            int((raw["value"] == 0).sum()),
            int(raw["value"].min()),
            int(raw["value"].max()),
        ],
    }
)
print(checks.to_string(index=False))

         점검 항목  결과
           행 수 262
         결측치 수   0
       중복 날짜 수   0
주 간격이 7일이 아닌 곳   0
    일요일이 아닌 날짜   0
       값이 0인 주   0
         값 최솟값  11
         값 최댓값 100


## 3. 마지막 주 처리

수집일(2026-11-09)에 끝나지 않았을 수 있는 마지막 주(2026-11-08)를 분석용 데이터에서 제외한다. 원본 파일은 바꾸지 않는다.

In [4]:
raw["is_partial"] = raw["date"] == raw["date"].max()
data = raw[~raw["is_partial"]].drop(columns="is_partial").reset_index(drop=True)

print("전체 :", len(raw), "주")
print("분석 :", len(data), "주")
print("제외한 주")
print(raw.loc[raw["is_partial"], ["date", "value"]])

전체 : 262 주
분석 : 261 주
제외한 주
          date  value
261 2026-11-08     74


## 4. 이상치 표시

- 각 주의 값에서 앞뒤 13주(약 한 분기)의 중앙값을 뺀 편차를 구한다.
- 편차가 중앙값 절대 편차(MAD) 기준으로 크게 벗어나면(로버스트 z 점수 절댓값 3.5 초과) 이상치로 표시한다.
- 표시만 하고 삭제하지 않는다.

In [5]:
WINDOW = 13        # 13주 = 약 한 분기
THRESHOLD = 3.5    # 로버스트 z 점수 기준

data["median_13w"] = data["value"].rolling(WINDOW, center=True, min_periods=7).median()
data["deviation"] = data["value"] - data["median_13w"]

center = data["deviation"].median()
mad = (data["deviation"] - center).abs().median()
assert mad > 0, "MAD가 0이면 이상치 기준을 다시 정해야 한다"

data["robust_z"] = 0.6745 * (data["deviation"] - center) / mad
data["is_outlier"] = data["robust_z"].abs() > THRESHOLD

print("MAD :", mad)
print("이상치로 표시된 주:", int(data["is_outlier"].sum()), "개")
print(data.loc[data["is_outlier"], ["date", "value", "median_13w", "robust_z"]].round(2).to_string(index=False))

MAD : 2.0
이상치로 표시된 주: 4 개
      date  value  median_13w  robust_z
2024-12-01     60        49.0      3.71
2026-10-11     73        40.5     10.96
2026-10-18    100        41.0     19.90
2026-10-25     82        57.0      8.43


## 5. 분석용 데이터 저장

원본은 `data/google_trends_overseas_travel_raw.csv`에 그대로 두고, 처리 결과는 별도 파일로 저장한다.

In [6]:
Path("data").mkdir(exist_ok=True)
OUT_PATH = "data/weekly_clean.csv"
data.to_csv(OUT_PATH, index=False, encoding="utf-8-sig")
print("저장:", OUT_PATH, data.shape)

저장: data/weekly_clean.csv (261, 6)
